In [50]:
"""
Task: Content Generation pipeline with Quality Control

Input:
- Topic
- Quality Requirements

Steps:
- Generate an initial draft
- Fact check the draft
- Improve the draft based on recommendations from the previous step
- Format for publication
"""

'\nTask: Content Generation pipeline with Quality Control\n\nInput:\n- Topic\n- Quality Requirements\n\nSteps:\n- Generate an initial draft\n- Fact check the draft\n- Improve the draft based on recommendations from the previous step\n- Format for publication\n'

In [51]:
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [52]:
class ContentState(TypedDict):
    topic: str
    requirements: str
    draft: str
    fact_check_results: str
    improved_content: str
    final_draft: str

In [53]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    max_retries=5,  # Automatically retry on 429 errors using exponential backoff
    timeout=60,
    temperature=0.2
)

In [54]:
def define_draft(state: ContentState):
    """Generate Draft about what is the topic tells depends on requirements"""
    prompt = f"""
        Write a 200-word blog post about : {state['topic']}

        Requirements: {state['requirements']}

        Focus on creating engaging, informative content
    """

    draft = llm.invoke(prompt).content

    print("=== STEP 1: Draft Generated ===")
    print(draft[:150])

    return {"draft": draft}


In [55]:
def check_draft(state: ContentState):
    """Check the draft generated by llm"""
    prompt = f"""
    Review the following blog post draft for factual accuracy and consistency:
    
    {state['draft']}

    Identify:
    1. Any factual claims that seem questionable
    2. Internal inconsistencies
    3. Statements that need citations

    Provide a brief report."""

    fact_check_results = llm.invoke(prompt).content

    print("=== STEP 2: Fact Check Complete ===")
    print(fact_check_results[:150])

    return {
        "fact_check_results": fact_check_results
    }

In [56]:
def improve_content(state: ContentState) -> ContentState:
    """Revise content based on fact check feedback"""

    prompt = f"""
    Here is a blog post draft:

    {state['draft']}

    Here is feedback from check_draft:

    {state['fact_check_results']}

    Revise the blog post to address the feedback while maintaining engaging writing. Keep it around 200 words."""

    improved = llm.invoke(prompt).content

    print("=== STEP 3: Content Improved ===")
    print(improved[:150])

    return {
        "improved_content": improved
    }

In [57]:
def format_output(state: ContentState) -> ContentState:
    """Format content with HTML tags and elements"""

    prompt = f"""
    Format the following blog post for web publication:

    {state['improved_content']}

    Add:
    - An engaging title wrapped in <h1> tags
    - Subheadings where appropriate with <h2> tags
    - Paragraph tags <p>
    - A meta description (1-2 sentences)

    Output the formatted HTML."""

    final = llm.invoke(prompt).content

    print("=== STEP 4: Formatted for Publication ===")
    print(final[:200])

    return {
        "final_draft": final
    }


In [58]:
graph = StateGraph(ContentState)

graph.add_node("define_draft", define_draft)
graph.add_node("check_draft", check_draft)
graph.add_node("improve_content", improve_content)
graph.add_node("format_output", format_output)

graph.add_edge(START, "define_draft")
graph.add_edge("define_draft", "check_draft")
graph.add_edge("check_draft", "improve_content")
graph.add_edge("improve_content", "format_output")
graph.add_edge("format_output", END)

app = graph.compile()

In [59]:
png_data = app.get_graph().draw_mermaid_png()
with open("graph.png", "wb") as f:
    f.write(png_data)

In [60]:
result = app.invoke({"topic": "what is YOLO, and give a pytorch code", 
                     "requirements": "Target audience: ai engineers"})

/home/etman/etman/WriterAgent/venv/lib64/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== STEP 1: Draft Generated ===
[{'type': 'text', 'text': "# Real-Time Detection: Demystifying YOLO with PyTorch\n\nIn computer vision, **YOLO (You Only Look Once)** revolutionized object detection by replacing slow, two-stage region-proposal networks (like Faster R-CNN) with a unified, single-stage framework. Instead of inspecting an image multiple times, YOLO treats object detection as a single, end-to-end regression problem.\n\nThe network processes the full image in a single forward pass. It divides the input into an $S \\times S$ grid, where each grid cell simultaneously predicts bounding box coordinates, confidence scores, and class probabilities. By assessing the entire global context at once, YOLO achieves incredible inference speeds with minimal background false-positives—making it the ideal choice for real-time edge deployment, robotics, and video analytics.\n\n### PyTorch Implementation\n\nYou can quickly deploy a pretrained YOLO model in PyTorch using `torch.hub`:\n\n```pyt

/home/etman/etman/WriterAgent/venv/lib64/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== STEP 2: Fact Check Complete ===
[{'type': 'text', 'text': 'Here is a review of the blog post draft regarding factual accuracy, internal consistency, and citation requirements:\n\n---\n\n### **Fact-Checking & Consistency Review Report**\n\n#### **1. Factual Accuracy**\n* **$S \\times S$ Grid Mechanism vs. Modern Implementations:** \n  * *Context:* The second paragraph describes YOLO dividing the input image into an $S \\times S$ grid to predict bounding boxes and probabilities.\n  * *Fact Check:* While this is factually correct for the original **YOLOv1** architecture (Redmon et al., 2016), modern YOLO versions (such as YOLOv5, YOLOv8, and YOLOv10 referenced later in the post) no longer rely on a simple single $S \\times S$ grid. Instead, they use multi-scale feature maps (Feature Pyramid Networks / PANet) operating across different strides (e.g., $80 \\times 80$, $40 \\times 40$, $20 \\times 20$) and modern anchor-free or optimized anchor mechanisms.\n  * *Recommendation:* Clarify 

/home/etman/etman/WriterAgent/venv/lib64/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== STEP 3: Content Improved ===
[{'type': 'text', 'text': "# Real-Time Detection: Demystifying YOLO with PyTorch\n\nIn computer vision, **YOLO (You Only Look Once)** revolutionized object detection by replacing slow, two-stage region-proposal networks with a unified, single-stage framework. Pioneered by [Redmon et al. (2016)](https://arxiv.org/abs/1506.02640), the original architecture introduced dividing images into an $S \\times S$ grid to predict bounding box coordinates and class probabilities in a single forward pass. \n\nWhile modern iterations have evolved beyond a simple grid to use multi-scale feature pyramids and anchor-free designs, YOLO's core philosophy—assessing the full global context instantly—remains unchanged, delivering high inference speeds with minimal background false-positives.\n\n### PyTorch Implementation\n\nToday, deploying YOLO is seamless. Using [Ultralytics](https://github.com/ultralytics/yolov5)' PyTorch integration via `torch.hub`, you can run a pretrain

/home/etman/etman/WriterAgent/venv/lib64/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== STEP 4: Formatted for Publication ===
[{'type': 'text', 'text': '```html\n<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <meta name="description" content="Discover how YOLO revolutionized real-time object detection in computer vision and learn how to implement a pretrained model in PyTorch with just a few lines of code.">\n    <title>Real-Time Object Detection: Demystifying YOLO with PyTorch</title>\n</head>\n<body>\n    <article>\n        <h1>Real-Time Object Detection: Demystifying YOLO with PyTorch</h1>\n\n        <p>In computer vision, <strong>YOLO (You Only Look Once)</strong> revolutionized object detection by replacing slow, two-stage region-proposal networks with a unified, single-stage framework. Pioneered by <a href="https://arxiv.org/abs/1506.02640" target="_blank" rel="noopener noreferrer">Redmon et al. (2016)</a>, the original architecture introduced dividing images 

In [61]:
print("\n" + "="*50)
print("FINAL RESULT")
print("="*50)
print(result["final_draft"])


FINAL RESULT
[{'type': 'text', 'text': '```html\n<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <meta name="description" content="Discover how YOLO revolutionized real-time object detection in computer vision and learn how to implement a pretrained model in PyTorch with just a few lines of code.">\n    <title>Real-Time Object Detection: Demystifying YOLO with PyTorch</title>\n</head>\n<body>\n    <article>\n        <h1>Real-Time Object Detection: Demystifying YOLO with PyTorch</h1>\n\n        <p>In computer vision, <strong>YOLO (You Only Look Once)</strong> revolutionized object detection by replacing slow, two-stage region-proposal networks with a unified, single-stage framework. Pioneered by <a href="https://arxiv.org/abs/1506.02640" target="_blank" rel="noopener noreferrer">Redmon et al. (2016)</a>, the original architecture introduced dividing images into an <i>S</i> &times; <i>

In [62]:
#html file part:
import webbrowser
import os

html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Content Generation Result</title>
    <style>
        body {{
            font-family: Georgia, serif;
            max-width: 800px;
            margin: 60px auto;
            padding: 0 20px;
            line-height: 1.7;
            color: #333;
            background: #fafafa;
        }}
        .section {{
            border: 1px solid #ddd;
            border-radius: 8px;
            padding: 20px 28px;
            margin-bottom: 32px;
            background: white;
        }}
        .section h2 {{
            font-size: 13px;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 0.08em;
            color: #888;
            margin: 0 0 14px;
        }}
        .section pre {{
            white-space: pre-wrap;
            font-family: Georgia, serif;
            margin: 0;
            font-size: 15px;
        }}
    </style>
</head>
<body>
    <div class="section">
        <h2>Topic</h2>
        <pre>{result["topic"]}</pre>
    </div>
    <div class="section">
        <h2>Requirements</h2>
        <pre>{result["requirements"]}</pre>
    </div>
    <div class="section">
        <h2>Draft</h2>
        <pre>{result["draft"]}</pre>
    </div>
    <div class="section">
        <h2>Fact Check Results</h2>
        <pre>{result["fact_check_results"]}</pre>
    </div>
    <div class="section">
        <h2>Improved Content</h2>
        <pre>{result["improved_content"]}</pre>
    </div>
    <div class="section">
        <h2>Final Draft</h2>
        {result["final_draft"]}
    </div>
</body>
</html>"""

output_path = os.path.abspath("output.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_content)

webbrowser.open(f"{output_path}")
print(f"\nOpened in browser: {output_path}")


Opened in browser: /home/etman/etman/WriterAgent/notebooks/output.html


Opening in existing browser session.
